# FinalGeo confirmatory RQ2 — T4 x2

Locked comparison of Uniform, PureGeo, and FinalGeo on seeds 3,4,5. The test set remains sealed until all nine epoch-100 checkpoints exist. Training uses the same 50+50 weights-only extension protocol as RQ2-v1.

## Secure checkout and hardware
Create Kaggle secret `github_token`, enable Internet, and select GPU T4 x2. Attach the completed `test-rq2` output and the saved FinalGeo frozen-protocol output.

In [ ]:
import os, subprocess, sys, time, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Kaggle secret github_token is missing'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy()
env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN_RUNTIME': github_token})
try:
    command = ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'] if (PROJECT_ROOT / '.git').is_dir() else ['git', 'clone', 'https://github.com/duyh80456-code/new-pruning.git', str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    env.pop('GITHUB_TOKEN_RUNTIME', None)
    github_token = None
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
import torch
assert torch.cuda.device_count() == 2, f'Expected T4x2, detected {torch.cuda.device_count()} GPU(s)'
print([torch.cuda.get_device_name(index) for index in range(2)])

## Locate and validate development evidence plus frozen selector

In [ ]:
import importlib, json
import rq2_anchor_placement
import rq2_parameter_exposure
import rq2_finalgeo_selector
import finalgeo_confirmatory
import scripts.run_finalgeo_confirmatory as run_finalgeo_confirmatory
rq2_anchor_placement = importlib.reload(rq2_anchor_placement)
rq2_parameter_exposure = importlib.reload(rq2_parameter_exposure)
rq2_finalgeo_selector = importlib.reload(rq2_finalgeo_selector)
finalgeo_confirmatory = importlib.reload(finalgeo_confirmatory)
run_finalgeo_confirmatory = importlib.reload(run_finalgeo_confirmatory)

RQ2_INPUT = Path('/kaggle/input/notebooks/dyhngg/test-rq2')
assert RQ2_INPUT.exists(), f'Attach RQ2-v1 output: {RQ2_INPUT}'
RQ2_ROOT = rq2_anchor_placement.find_rq2_development_root(
    RQ2_INPUT, '/kaggle/working/materialized-rq2-final-confirmatory'
)
freeze_candidates = sorted(Path('/kaggle/input').rglob('finalgeo_frozen_protocol.json'))
if len(freeze_candidates) > 1:
    raise RuntimeError(f'Expected at most one frozen protocol, found: {freeze_candidates}')
if freeze_candidates:
    FREEZE_DIR = freeze_candidates[0].parent
    print('Using attached frozen protocol:', FREEZE_DIR)
else:
    # Safe only because this cell runs before any seed-3/4/5 training or test evaluation.
    FREEZE_DIR = Path('/kaggle/working/finalgeo-freeze-before-confirmatory')
    rq2_finalgeo_selector.freeze_finalgeo_selector(RQ2_ROOT, FREEZE_DIR, PROJECT_ROOT)
    print('No attached freeze artifact; deterministically froze FinalGeo before training:', FREEZE_DIR)
protocol = finalgeo_confirmatory.validate_frozen_protocol(
    RQ2_ROOT, FREEZE_DIR, PROJECT_ROOT, verify_git=True
)
print('Frozen FinalGeo:', protocol['finalgeo_anchors'])
print('Freeze ID:', protocol['freeze_id_sha256'])
print('Confirmatory seeds:', protocol['seeds_confirmatory'])

## Run all nine models
Two jobs run concurrently. Rerunning this cell in the same persisted working directory resumes/skips completed epoch checkpoints. No result-dependent early stop is allowed.

In [ ]:
RUN_DIR = Path('/kaggle/working/finalgeo-confirmatory')
started = time.perf_counter()
result = run_finalgeo_confirmatory.run_confirmatory(
    development_root=RQ2_ROOT,
    freeze_dir=FREEZE_DIR,
    output_dir=RUN_DIR,
    gpu_ids=[0, 1],
)
print(f'Completed in {(time.perf_counter() - started) / 3600:.2f} hours')
print(json.dumps(result['decision'], indent=2))

## Inspect the preregistered result

In [ ]:
import pandas as pd
from IPython.display import display
display(pd.read_csv(RUN_DIR / 'finalgeo_method_summary.csv'))
display(pd.read_csv(RUN_DIR / 'finalgeo_seed_comparison.csv'))
display(json.loads((RUN_DIR / 'finalgeo_confirmatory_decision.json').read_text()))

## Validate and export a compact resumable bundle

In [ ]:
required = [
    RUN_DIR / 'finalgeo_confirmatory_decision.json',
    RUN_DIR / 'finalgeo_dense_metrics_all.csv',
    RUN_DIR / 'finalgeo_geometry_all.csv',
    RUN_DIR / 'finalgeo_method_summary.csv',
    RUN_DIR / 'finalgeo_seed_comparison.csv',
    *[RUN_DIR / method / f'seed_{seed}' / 'checkpoint.pt' for method in ['uniform','puregeo','finalgeo'] for seed in [3,4,5]],
]
missing = [str(path) for path in required if not path.is_file() or path.stat().st_size == 0]
assert not missing, f'Missing outputs: {missing}'
bundle_path = Path('/kaggle/working/finalgeo-confirmatory-results.zip')
include = []
include += [path for path in (RUN_DIR / 'protocol').rglob('*') if path.is_file()]
include += [RUN_DIR / 'resolved_config.yaml', RUN_DIR / 'finalgeo_confirmatory_decision.json',
            RUN_DIR / 'finalgeo_dense_metrics_all.csv', RUN_DIR / 'finalgeo_geometry_all.csv',
            RUN_DIR / 'finalgeo_method_summary.csv', RUN_DIR / 'finalgeo_seed_comparison.csv',
            RUN_DIR / 'finalgeo_runtime_by_job.csv']
for method in ['uniform','puregeo','finalgeo']:
    for seed in [3,4,5]:
        include += [RUN_DIR / method / f'seed_{seed}' / name for name in
                    ['checkpoint.pt','training_metrics_1_100.csv','training_provenance.json']]
        include += [RUN_DIR / 'evaluation' / method / f'seed_{seed}' / name for name in
                    ['budget_metrics.csv','representation_local_geometry.csv']]
        include += [RUN_DIR / 'predictions' / method / f'seed_{seed}' / 'predictions_all_widths.csv']
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as bundle:
    for path in include:
        assert path.is_file(), f'Missing bundle artifact: {path}'
        bundle.write(path, path.relative_to(RUN_DIR))
print('Download/persist:', bundle_path, f'{bundle_path.stat().st_size / 2**30:.2f} GiB')
bundle_path